# Velocity Model Smoothing Demo
Builds a time velocity model from `velan` picks with Seismic Unix (`unisam` along time, then `unisam2` across CDPs), smooths it with pyseiskit's moving average (box) and checks it against the same box built from SU programs. SU's own `smooth2` and `velconv` are shown for reference.

Requires Seismic Unix (`$CWPROOT/bin` in `PATH`).

In [ ]:
%pip install bokeh segyio

# *** Install pyseiskit from PyPI
# %pip install pyseiskit
# *** Install pyseiskit for local development
# %pip install -e ..
!uv pip install -e .. --reinstall

## 1. Data Setup
Needs a `.su` dataset and its `velan` picks file (`cdp=`, then `tnmo=`/`vnmo=` per CDP) in this folder. The dataset is only read for its geometry (CDP range and time sampling).

In [ ]:
import segyio
import numpy as np
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import row
from bokeh.models import LinearColorMapper, ColorBar
from velanPicksReader import readVelanPicks
from seismicUnix import isSeismicUnixAvailable, runSeismicUnix
from pyseiskit import velocityModel
from pyseiskit import PALETTES
output_notebook()

if not isSeismicUnixAvailable():
    raise RuntimeError("Seismic Unix programs were not found: add $CWPROOT/bin to PATH.")

# Local files only: keep them in the 'demos' folder, next to this notebook
datasetFilename = 'line-1-line-2-data001_crsRegGathers_cdps.su'
picksFilename = 'line-1-line-2-data001_crsRegGathers_cdps.picksvelan.dat'

velocityPicks = readVelanPicks(picksFilename)
pickedCdps = np.array(sorted(velocityPicks))
pickingInterval = int(pickedCdps[1] - pickedCdps[0])

# Model geometry from the dataset headers: one column per CDP, same time sampling as the data
with segyio.su.open(datasetFilename, "r", ignore_geometry=True, endian='little') as seismicDataset:
    datasetCdps = seismicDataset.attributes(segyio.TraceField.CDP)[:]
    numberSamples = len(seismicDataset.samples)
    intervalTimeSamples = seismicDataset.header[0][segyio.TraceField.TRACE_SAMPLE_INTERVAL] / 1e6

cdps = np.arange(datasetCdps.min(), datasetCdps.max() + 1)
timeSamples = np.arange(numberSamples) * intervalTimeSamples

print(f"{len(pickedCdps)} picked CDPs from {pickedCdps[0]} to {pickedCdps[-1]}, every {pickingInterval} CDPs.")
print(f"Model: {numberSamples} samples every {intervalTimeSamples} s, CDPs {cdps[0]} to {cdps[-1]}.")

def formatValues(values):
    return ",".join(repr(float(value)) for value in values)

def plotVelocityModel(model, title, verticalSamples, velocityRange, verticalLabel="Time (s)", width=420):
    plotFigure = figure(
        width=width,
        height=500,
        title=title,
        x_axis_label="CDP",
        y_axis_label=verticalLabel,
        x_range=(cdps[0], cdps[-1]),
        y_range=(verticalSamples[-1], verticalSamples[0])
    )
    colorMapper = LinearColorMapper(palette=PALETTES['jet'], low=velocityRange[0], high=velocityRange[1])
    plotFigure.image(
        image=[model],
        x=cdps[0],
        y=verticalSamples[0],
        dw=cdps[-1] - cdps[0],
        dh=verticalSamples[-1] - verticalSamples[0],
        color_mapper=colorMapper
    )
    plotFigure.add_layout(ColorBar(color_mapper=colorMapper, title="m/s"), 'right')
    return plotFigure

## 2. Build the Velocity Model (SU `unisam` + `unisam2`)
`unisam` interpolates each picked CDP along time, and `unisam2` interpolates across CDPs (`unisam2` applies `method1=` across CDPs, the opposite of its self-documentation). CDPs outside the picked range repeat the nearest picked CDP.

In [ ]:
# One velocity function per picked CDP [2D: samples x picked cdps]
pickedVelocityFunctions = runSeismicUnix(
    " && ".join(
        f"unisam xin={formatValues(velocityPicks[cdp]['times'])} yin={formatValues(velocityPicks[cdp]['velocity'])}"
        f" nout={numberSamples} dxout={intervalTimeSamples} method=linear >> output.bin"
        for cdp in pickedCdps
    ),
    numberSamples
)

def interpolateAcrossCdps(velocityFunctions):
    return runSeismicUnix(
        f"unisam2 < input.bin nx1={numberSamples} dx1={intervalTimeSamples} n1={numberSamples} d1={intervalTimeSamples} f1=0"
        f" x2={formatValues(pickedCdps)} n2={len(cdps)} f2={cdps[0]} d2=1 method1=linear method2=linear > output.bin",
        numberSamples,
        velocityFunctions
    )

rmsVelocityModel = interpolateAcrossCdps(pickedVelocityFunctions)
velocityRange = (rmsVelocityModel.min(), rmsVelocityModel.max())

modelFigure = plotVelocityModel(rmsVelocityModel, "RMS Velocity (unisam + unisam2)", timeSamples, velocityRange, width=800)
pickedTimes = [time for cdp in pickedCdps for time in velocityPicks[cdp]['times']]
pickedTimeCdps = [cdp for cdp in pickedCdps for _ in velocityPicks[cdp]['times']]
modelFigure.scatter(pickedTimeCdps, pickedTimes, color='white', size=3)
show(modelFigure)

## 3. Moving Average Smoothing
A box of `timeWindow` seconds x `cdpWindow` CDPs, centred on each sample and truncated at the model edges (only the samples inside the model are averaged).

### 3A. On the Whole Model
### 3B. On the Picked CDPs, then Across CDPs
Smooths only the picked velocity functions (one column every `pickingInterval` CDPs, so `cdpWindow` counts picked CDPs) and runs `unisam2` afterwards.

In [ ]:
# 3A. 0.2 s x 201 CDPs box on the whole model
movingAverageModel = velocityModel.applyMovingAverageSmoothing(
    rmsVelocityModel,
    timeWindow=0.2,
    cdpWindow=201,
    intervalTimeSamples=intervalTimeSamples
)

# 3B. 0.2 s x 3 picked CDPs box, then unisam2 across CDPs
smoothedPickedFunctions = velocityModel.applyMovingAverageSmoothing(
    pickedVelocityFunctions,
    timeWindow=0.2,
    cdpWindow=3 * pickingInterval,
    intervalTimeSamples=intervalTimeSamples,
    intervalCdps=pickingInterval
)
pickedMovingAverageModel = interpolateAcrossCdps(smoothedPickedFunctions)

show(
    row(
        plotVelocityModel(rmsVelocityModel, "Unsmoothed", timeSamples, velocityRange),
        plotVelocityModel(movingAverageModel, "Moving Average (whole model)", timeSamples, velocityRange),
        plotVelocityModel(pickedMovingAverageModel, "Moving Average (picked CDPs)", timeSamples, velocityRange)
    )
)

## 4. Benchmark Against Seismic Unix
No SU program computes a 2D box mean, so the reference is a chain of trace programs (see the comments in `movingAverageWithSeismicUnix`). Windows are converted to half box sizes with the same rule as `applyMovingAverageSmoothing`. SU computes in 32-bit floats and pyseiskit in 64-bit floats, so differences around 1e-6 of the velocities are rounding.

In [ ]:
import time

def movingAverageWithSeismicUnix(model, halfTimeSamples, halfCdps):
    """
    SU box mean: suconv (time) + sumix (CDPs) box sums of the model, divided by the
    same box sums of an all-ones model (number of samples inside the truncated box).
    """
    timeBox = ",".join(["1"] * (2 * halfTimeSamples + 1))
    cdpBox = ",".join(["1"] * (2 * halfCdps + 1))

    # *** suconv is causal and appends 2*halfTimeSamples samples: suwind recentres it.
    # *** sumix is a trailing mix that skips its first traces and divides by the number of weights:
    # *** null traces on both sides, dropping the first 2*halfCdps outputs and sugain undo that.
    boxSum = (
        "boxSum() { "
        f"suconv < $1 filter={timeBox} | suwind itmin={halfTimeSamples} itmax={halfTimeSamples + numberSamples - 1} "
        "| sushw key=delrt a=0 > timeSum.su; "
        f"sunull nt={numberSamples} ntr={halfCdps} dt={intervalTimeSamples} > nullTraces.su; "
        f"cat nullTraces.su timeSum.su nullTraces.su | sumix mix={cdpBox} > mixed.su; "
        f"suwind < mixed.su skip={2 * halfCdps} | sugain scale={2 * halfCdps + 1} > $2; "
        "}; "
    )
    command = (
        boxSum
        + f"suaddhead ns={numberSamples} < input.bin | sushw key=dt a={round(intervalTimeSamples * 1e6)} > model.su && "
        + "sugain scale=0 < model.su | sugain bias=1 > ones.su && "
        + "boxSum model.su sums.su && boxSum ones.su counts.su && "
        + "suop2 sums.su counts.su op=quo | sustrip head=/dev/null outpar=/dev/null > output.bin"
    )
    return runSeismicUnix(command, numberSamples, model)

benchmarkCases = [
    ('picked CDPs', pickedVelocityFunctions, 0.2, 3 * pickingInterval, pickingInterval),
    ('picked CDPs', pickedVelocityFunctions, 0.5, 5 * pickingInterval, pickingInterval),
    ('whole model', rmsVelocityModel, 0.2, 51, 1),
    ('whole model', rmsVelocityModel, 0.4, 201, 1),
]

print(f"{'case':<40} {'max diff (m/s)':>15} {'max diff / max v':>17} {'pyseiskit (s)':>14} {'SU (s)':>9}")
for modelName, model, timeWindow, cdpWindow, intervalCdps in benchmarkCases:
    startTime = time.perf_counter()
    pyseiskitResult = velocityModel.applyMovingAverageSmoothing(model, timeWindow, cdpWindow, intervalTimeSamples, intervalCdps)
    pyseiskitSeconds = time.perf_counter() - startTime

    halfTimeSamples = round(timeWindow / intervalTimeSamples) // 2
    halfCdps = round(cdpWindow / intervalCdps) // 2
    startTime = time.perf_counter()
    seismicUnixResult = movingAverageWithSeismicUnix(model, halfTimeSamples, halfCdps)
    seismicUnixSeconds = time.perf_counter() - startTime

    maxDifference = np.abs(pyseiskitResult - seismicUnixResult).max()
    print(
        f"{modelName + f' {timeWindow} s x {cdpWindow} CDPs':<40} {maxDifference:>15.3e}"
        f" {maxDifference / np.abs(seismicUnixResult).max():>17.3e} {pyseiskitSeconds:>14.4f} {seismicUnixSeconds:>9.4f}"
    )

## 5. Comparison with SU `smooth2`
SU's usual velocity smoother is `smooth2` (damped least squares, `r1=` along time, `r2=` across CDPs), run here directly from SU next to the moving average.

In [ ]:
smooth2Model = runSeismicUnix(
    f"smooth2 < input.bin n1={numberSamples} n2={len(cdps)} r1=10 r2=20 > output.bin",
    numberSamples,
    rmsVelocityModel
)

selectedCdp = int(pickedCdps[len(pickedCdps) // 2])
selectedColumn = int(np.searchsorted(cdps, selectedCdp))
functionsFigure = figure(
    width=420,
    height=500,
    title=f"Velocity at CDP {selectedCdp}",
    x_axis_label="Velocity (m/s)",
    y_axis_label="Time (s)",
    y_range=(timeSamples[-1], timeSamples[0])
)
for label, model, color in [
    ('unsmoothed', rmsVelocityModel, 'black'),
    ('moving average (whole model)', movingAverageModel, 'red'),
    ('moving average (picked CDPs)', pickedMovingAverageModel, 'blue'),
    ('smooth2 r1=10 r2=20', smooth2Model, 'green'),
]:
    functionsFigure.line(model[:, selectedColumn], timeSamples, legend_label=label, color=color, line_width=1.5)
functionsFigure.scatter(velocityPicks[selectedCdp]['velocity'], velocityPicks[selectedCdp]['times'], legend_label='picks', color='black', size=7)
functionsFigure.legend.location = 'bottom_left'

show(row(plotVelocityModel(smooth2Model, "smooth2 r1=10 r2=20", timeSamples, velocityRange), functionsFigure))

## 6. Interval Velocity in Depth (SU `velconv`)
The smoothed RMS model converted with `velconv intype=vrmst outtype=vintz` (Dix, then time to depth). Dix amplifies RMS velocity kinks, so it runs on a smoothed model.

In [ ]:
intervalDepthSamples = 10.0
numberDepthSamples = 800
depthSamples = np.arange(numberDepthSamples) * intervalDepthSamples

# *** velconv seeks its input, so it reads a file instead of a pipe
depthVelocityModel = runSeismicUnix(
    f"velconv < input.bin intype=vrmst outtype=vintz nt={numberSamples} dt={intervalTimeSamples}"
    f" nz={numberDepthSamples} dz={intervalDepthSamples} > output.bin",
    numberDepthSamples,
    movingAverageModel
)

show(
    row(
        plotVelocityModel(movingAverageModel, "RMS Velocity (moving average)", timeSamples, velocityRange),
        plotVelocityModel(
            depthVelocityModel,
            "Interval Velocity in Depth",
            depthSamples,
            (depthVelocityModel.min(), depthVelocityModel.max()),
            verticalLabel="Depth (m)"
        )
    )
)